# ETL Pipeline using AWS RDS

**Topic:** 09 Data Analysis Process · **Level:** 🟠 Intermediate · **Type:** COLLAB (Extra Credit)

## What this notebook is

This is a complete, runnable **ETL pipeline** — Extract, Transform, Load — applied to the IPL cricket dataset (`matches.csv` and `deliveries.csv`, covering 2008–2020) and ending in a live MySQL database hosted on **AWS RDS**. It is the rare notebook that goes all the way around the data lifecycle: pull files, clean them, enrich them with a scoring system, aggregate them into summary tables, push four tables into a cloud database, and then *query the database back* to prove the round trip worked. The accent is two-fold: a disciplined three-phase structure (Extract / Transform / Load) and a concrete cloud deployment on Amazon's managed MySQL service.

## The shape of the journey

**Extract** loads the two CSVs and inspects them. **Transform** is the expressive middle: missing values are dropped or filled, `umpire3` is discarded, ball-by-ball deliveries are scored with the **Dream11 point system** (batting points, bowling economy and wicket bonuses, fielding points) and aggregated per player per match, both sides of the game get summary tables, and they are merged with match metadata enriched with a playing-XI bonus. **Load** connects to an AWS RDS MySQL instance, creates `matches`, `deliveries`, `dream11_points` and `player_stats` tables, and streams the DataFrames with `to_sql`. Finally, analytical queries run against the database — top batsmen, top Dream11 scorers, best economy bowlers, matches per season — closing the loop.

## What to notice

The notebook's quiet lessons are worth as much as the code: a connection string in `sqlalchemy`'s `mysql+pymysql://` form, chunked `to_sql` for big tables, `if_exists='replace'` vs `'append'` discipline, and the habit of verifying every step with a printed shape. By the end you have seen ETL as a loop, not a list.


## What is ETL?

**ETL** stands for **Extract, Transform, Load** — the standard pipeline that moves raw data from source systems into a single, consistent store where analysis actually happens. The notebook's diagram captures it as three boxes in a line: **Extract** pulls data from wherever it lives, **Transform** repairs and reshapes it, and **Load** writes the result to a destination.

Each phase has a job. **Extract** is about *getting hold* of the data without damaging it — reading a CSV, calling an API, pulling from a database, or scraping a page — and keeping the original intact. **Transform** is where value is added: cleaning (removing duplicates, handling nulls), filtering (keeping only relevant rows and columns), aggregating (grouping to sums and means), feature engineering (creating new columns), and validating the result. **Load** is the final write to a **data warehouse**, **database** or **data lake**, choosing between a full load and an incremental one.

The mental model to keep: ETL is a *conveyor belt*, not a one-off script. Sources change, so the belt must be re-runnable; transforms change, so each stage must be inspectable; and the whole point is that analysts downstream query a clean, consistent store instead of chasing file formats.


## The Three Steps

ETL decomposes into three responsibilities, and this notebook will honour them as literal phases. **Extract** — bring data from source systems (CSV, API, database, web) and keep the original column intact. Here that means two Kaggle CSVs: `matches.csv` for match-level facts and `deliveries.csv` for ball-by-ball detail.

**Transform** — this is where the data becomes *analysable*: clean it (remove duplicates, handle nulls), filter to the rows and columns that matter, aggregate with `groupby` / `sum` / `mean`, engineer new features, and validate quality. In this notebook, Transform produces the Dream11 points engine and the per-player summaries — the entire analytical payload of the dataset.

**Load** — write the transformed result to its destination: a data warehouse, a database, or a data lake. The destination here is an **AWS RDS** MySQL instance, and the code must choose full versus incremental loading per table. Read these three phases as a contract: extract something worth keeping, transform it until it is trustworthy, load it where it can be queried.


## Why ETL?

A pipeline earns its keep by answering six needs. **Data integration** — one dataset rarely tells the whole story; ETL combines tables from multiple sources into one view. In this project, match facts and ball-by-ball events become one enriched player table. **Data quality** — because data is cleaned and validated before anyone queried it, the downstream consumer never has to hope. **Historical analysis** — a stored store supports trend analysis over time, exactly what the IPL's 2008–2020 span enables.

**Reporting** — BI dashboards and analytics draw from one consistent schema rather than ad-hoc files. **Machine learning** — models need prepared, typed features, which is precisely the output of a clean transform stage. **Compliance and governance** — a governed store means you can say where data came from, how it changed, and who touched it.

The unifying idea: ETL converts *data* into *assets*. A CSV is something you load and pray about; a loaded, versioned store is something you can query, audit, and trust. Everything downstream — dashboards, models, reports — stands on the quality the pipeline established earlier.


## What is AWS RDS?

**Amazon RDS (Relational Database Service)** is a managed cloud database: Amazon provisions, runs, and maintains a relational database for you — backups, patching, monitoring, failover — so you focus on queries, not server upkeep. This notebook uses it as the landing zone for the transformed IPL data, running **MySQL** under the hood.

The service supports the major engines: **MySQL** (the most popular open-source database), **PostgreSQL** (advanced open-source), **MariaDB** (a MySQL fork), **Oracle** and **SQL Server** (enterprise picks), and **Amazon Aurora** (an AWS-optimized MySQL/PostgreSQL-compatible engine).

Why choose it? **Managed** — backups, patching, monitoring handled by AWS. **Scalable** — storage and compute resize without rebuilding. **High availability** — Multi-AZ deployments survive instance failure. **Security** — encryption, VPC isolation, IAM authentication. **Cost** — you pay for what you use, and the Free Tier makes this exercise effectively free. In short, RDS gives you a production-grade database with none of the night shift.


## Setting Up AWS RDS (Step-by-Step)

Before any code runs, a database must exist in the cloud, and this notebook walks the full setup. **Prerequisites**: an AWS account (Free Tier eligible) and a SQL client such as MySQL Workbench for eyeballing results.

**Steps**: login to the AWS Console, search for RDS, and click *Create Database*; choose the **MySQL** engine and the **Free Tier** template; configure the instance — DB identifier `ipl-database`, master username `admin`, a master password of your choosing; enable **Publicly Accessible** for this test setup; add a security-group rule opening port `3306`; and create the database. AWS then provisions the instance and hands back a connection block that matters more than everything else:

- **Endpoint** — `ipl-database.xxxxx.sa-east-1.rds.amazonaws.com`
- **Port** — `3306`
- **Database** — `ipl_db`
- **Username / password** — `admin` / your own

Copy those exact values into the Python config later; they are the keys to every connection in the rest of the notebook. Treat the endpoint like a hostname — the whole pipeline hinges on getting it into the code verbatim.


## Installation

The notebook begins by installing the libraries the whole pipeline depends on: `pip install pandas numpy mysql-connector-python sqlalchemy pymysql`. Each earns its place. `pandas` and `numpy` are the data machinery — reading the CSVs and computing the aggregations. `mysql-connector-python` is the official driver that lets Python talk to a MySQL server directly, used here for a quick connection test.

`sqlalchemy` is the crucial one for pandas users: it creates `engine` objects, and `df.to_sql(con=engine)` and `pd.read_sql(query, con=engine)` know how to use any engine from any supported database. `pymysql` is the pure-Python MySQL driver that SQLAlchemy actually uses to speak to the MySQL wire protocol when the connection string says `mysql+pymysql://`.

Note the architecture insight: pandas never talks to the database itself — it always goes through the engine. Install once, import at the top, and every later cell inherits the same plumbing. Keeping the install in its own cell also makes the run order explicit: if a dependency is missing, the notebook fails at the very start with a clear message instead of halfway through a SQL query.


In [ ]:
# Install required libraries# !pip install pandas numpy mysql-connector-python sqlalchemy pymysqlimport pandas as pdimport numpy as npimport mysql.connectorfrom sqlalchemy import create_engineimport warningswarnings.filterwarnings('ignore')print("Libraries imported successfully!")

## Phase 1 — Extract

Extract is the first phase and the least glamorous: get the raw data out of its source with its integrity intact. The principle hammered here is **keep the original untouched** — extraction reads data, it never mutates the source — because every later repair depends on being able to re-open the pristine file and re-verify.

Two files carry this project. **`matches.csv`** is match-level: one row per IPL match, carrying season, teams, toss, result, venue, and the winner. **`deliveries.csv`** is ball-level: one row for every delivery of every innings, naming batsman, bowler, runs scored, extras, and dismissals. They are a classic relational pair — the match is the parent, the hundreds of deliveries are its children, joined by `match_id`.

That split is the real lesson of Extract: *choose the grain* before you load. The finest grain (the ball) powers the Dream11 scoring; the coarser grain (the match) powers the season summaries. Both must be loaded, and the pipeline keeps both throughout Transform.


## Download the Dataset

The source for this project is the **IPL Complete Dataset (2008–2020)** on Kaggle — `https://www.kaggle.com/datasets/patrickb1912/ipl-complete-dataset-20082020`. Two files matter here: `matches.csv` covering every fixture with its metadata, and `deliveries.csv` covering every ball bowled across those seasons.

Two practical points. First, a Kaggle dataset is downloaded as a zip whose inner CSVs are large-ish but far from enormous — `deliveries.csv` is comfortably loadable with plain `pd.read_csv`, which is exactly what the Extract phase does. Second, keep the two files *together* in the runtime directory: the later `read_csv` calls expect `matches.csv` and `deliveries.csv` at the working path. If the verify prints shape mismatches (you saw `Matches shape:` and `Deliveries shape:` in the loader), re-check the download rather than the code.

Take the download itself seriously — the quality of every subsequent chart, score, and query rests on the raw files being the genuine Kaggle snapshots. Garbage in, garbage everywhere downstream.


## Reading the two source files

The Extract phase's first real code loads the pair of CSVs that feed the entire pipeline. `matches` carries match-level facts — one row per fixture with seasons, teams, venues, and winners — while `deliveries` carries ball-level detail: every delivery across every match, naming the batter, the bowler, the runs, the extras, and any dismissal.

The loader prints the two shapes side by side, which is the correct instinct: `Matches shape: (...)` and `Deliveries shape: (...)` are the first contract with the data. Deliveries should dwarf matches by orders of magnitude — thousands of balls versus hundreds of fixtures — and that size disparity is precisely why the later Load phase will stream `deliveries` in chunks while `matches` writes in one shot.

Compare the grain before moving on: the summary and scoring logic that follows is all *derived* from these two tables. If the shapes look wrong at this cell, everything downstream inherits the error — so verify the count, then proceed.


In [ ]:
# Load the datasetsmatches = pd.read_csv('matches.csv')deliveries = pd.read_csv('deliveries.csv')print(f"Matches shape: {matches.shape}")print(f"Deliveries shape: {deliveries.shape}")

In [ ]:
# Inspect matches datamatches.head()

In [ ]:
# Inspect deliveries datadeliveries.head()

In [ ]:
# Check columnsprint("Matches columns:")print(matches.columns.tolist())print("\nDeliveries columns:")print(deliveries.columns.tolist())

## Phase 2 — Transform

Transform is where raw data becomes *analysable* data, and its four responsibilities appear later as explicit steps. **Clean** missing values — the notebook drops a useless `umpire3` column and rows whose winner is unknown, then fills the delivery-table gaps with the sentinel `'None'`. **Filter** relevant columns and rows so only what matters survives to the destination.

**Feature engineering** is the star here: ball-by-ball deliveries are scored with the **Dream11 point system**, producing `batting_points`, `bowling_points`, `fielding_points` and a `total_points` per ball. **Aggregation** then collapses those per-ball scores into per-player-per-match summaries and, after merging in match metadata, into overall career statistics with a strike rate.

The ordering matters: clean first (so scoring never runs on NaN), then engineer, then aggregate — aggregates computed on dirty units would be dirty aggregates. Watch how every transformed table announces a `shape` after creation: Transform treats its own output as something to be inspected, not assumed.


## The Dream11 Point System

Dream11 is a fantasy-cricket platform, and its scoring rules are the business logic this transform implements — the reason the notebook can turn raw deliveries into points. The system has three budgets. **Batting**: each run earns +1; a boundary adds +1 bonus and a six adds +2; milestone bonuses land at 30, 50 and 100 runs (+4 / +8 / +16); and a duck (out for zero) costs −2. So a 50 with two sixes is worth materially more than a flat 50.

**Bowling**: a wicket is +25, with haul bonuses at 3, 4 and 5 wickets (+4 / +8 / +16); a maiden over is +12; and economy rewards tighten — under 5 an over earns +6, between 5 and 6 earns +4, between 6 and 7 earns +2. **Fielding and extras**: a catch is +8, a stumping or run-out +12, selection in the playing XI is +4 — and a wide or no-ball −1 rides against the bowler.

Hold this table in view as the Transform phase runs: every point a batsman or bowler accrues in the code traces back to one of these rows. Rules become code, code becomes aggregates, aggregates become the load phase.


## Step 1 — Clean Matches Data

Cleaning `matches` is short and pointed. First, `matches.isnull().sum()` audits where the match table leaks: the biggest gap is `umpire3`, a column that existed in early IPL seasons and was dropped later, leaving most rows empty. The repair is honest: `matches.drop(['umpire3'], axis=1, errors='ignore')` removes a column that is mostly missing rather than pretending to fill it.

Second, rows whose critical outcome is missing — no `winner` — are dropped with `dropna(subset=['winner'])`, because a match without a winner cannot feed any winner-based analysis downstream.

The lesson, in one line: **understand the missingness before you repair it**. Column missing everywhere → drop the column; row missing the key fact → drop the row. Both decisions come from the audit line `print("Missing values in matches:")`, and both are confirmed by re-printing the cleaned shape so nobody mistakes the intent for guesswork. The order matters too: drop the useless column before the rows, because addressing one categorical leak first keeps each decision attributable to the audit that uncovered it.


In [ ]:
# Check missing values in matchesprint("Missing values in matches:")print(matches.isnull().sum())

In [ ]:
# Drop columns with too many missing valuesmatches_clean = matches.drop(['umpire3'], axis=1, errors='ignore')# Drop rows with critical missing valuesmatches_clean = matches_clean.dropna(subset=['winner'])print(f"Cleaned matches shape: {matches_clean.shape}")

## Step 2 — Clean Deliveries Data

`deliveries` is the finer-grained table, and its missingness is concentrated in the three *dismissal* columns: `player_dismissed`, `dismissal_kind`, and `fielder` are blank for the overwhelming majority of balls — because most deliveries do not dismiss anyone. Filling them with the string `'None'` (not `NaN`) is a deliberate choice: the later scoring functions call `row['player_dismissed'] != 'None'` to detect a wicket, and they must be able to compare values without tripping over missing markers.

A working copy is involved — `deliveries_clean = deliveries.copy()` — so the raw frame stays pristine for re-auditing. The functions `calculate_batting_points`, `calculate_bowling_points` and `calculate_fielding_points` will then read these columns.

Note the difference from the matches repair: `matches` *dropped* empty columns, while `deliveries` *fills* them — because here emptiness is meaningful (no dismissal) rather than absence of a feature. Match the repair to the meaning of the missingness, and both tables survive cleaning intact.


In [ ]:
# Check missing values in deliveriesprint("Missing values in deliveries:")print(deliveries.isnull().sum())

In [ ]:
# Fill missing values in player_dismisseddeliveries_clean = deliveries.copy()deliveries_clean['player_dismissed'] = deliveries_clean['player_dismissed'].fillna('None')deliveries_clean['dismissal_kind'] = deliveries_clean['dismissal_kind'].fillna('None')deliveries_clean['fielder'] = deliveries_clean['fielder'].fillna('None')print(f"Cleaned deliveries shape: {deliveries_clean.shape}")

## Step 3 — Feature Engineering (Dream11 Points)

This is the heart of the transform: a DataFrame `apply` loop that turns every delivery's detail into its Dream11 contribution. `calculate_batting_points(row)` walks a single ball row with plain Python logic — start at zero, add the `batsman_runs`, add +1 if the ball was a four, +2 if it was a six — and the function's simplicity is the point.

The pattern: **write a tiny numbered function for one row, then apply it to every row**. `deliveries_clean.apply(calculate_batting_points, axis=1)` runs the function across rows (axis=1 = across the row's columns), producing a new series that becomes `batting_points`. The notebook prints the result side by side with `batsman` and `batsman_runs`, so you can verify the arithmetic by eye on real deliveries before it is trusted.

Why not a clever one-liner? Because a named function is *testable* — later readers can read the rules off the function, and a bug shows in one place instead of hiding inside a lambda chain.


In [ ]:
# Calculate batting points for each balldef calculate_batting_points(row):    points = 0    # Run points    points += row['batsman_runs']    # Boundary bonus    if row['batsman_runs'] == 4:        points += 1    elif row['batsman_runs'] == 6:        points += 2    return points# Apply functiondeliveries_clean['batting_points'] = deliveries_clean.apply(calculate_batting_points, axis=1)print("Batting points calculated!")deliveries_clean[['batsman', 'batsman_runs', 'batting_points']].head(10)

### Bowling Points Calculation

Bowling points encode two opposing forces. The reward side: if `player_dismissed != 'None'`, the ball took a wicket, worth **+25** points. The penalty side: the bowler absorbs the extras — a `wide_runs` over 0 costs −1 point and a `noball_runs` over 0 costs another −1, because in fantasy scoring a wide or a no-ball is the bowler's debt, not the batter's credit.

Look at the shape of `calculate_bowling_points`: a handful of `if` / `elif` checks on the row's columns, a running `points` total, and a return. It is row-level logic on explicitly typed columns — which is why the earlier cleaning step turned the empty dismissal fields into `'None'`: this function can now safely compare `row['player_dismissed'] != 'None'` on every single ball.

The wider point about feature engineering: a derived column (`bowling_points`) is only as trustworthy as the unit-level rules that produced it. Scoring on clean, typed columns — and inspecting the first ten rows afterwards — is how a fantasy metric stays defensible from its first ball to its final aggregate.


In [ ]:
# Calculate bowling points for each balldef calculate_bowling_points(row):    points = 0    # Wicket points    if row['player_dismissed'] != 'None':        points += 25    # Extra runs (wides and no-balls count against bowler)    if row['wide_runs'] > 0:        points -= 1    if row['noball_runs'] > 0:        points -= 1    return points# Apply functiondeliveries_clean['bowling_points'] = deliveries_clean.apply(calculate_bowling_points, axis=1)print("Bowling points calculated!")deliveries_clean[['bowler', 'player_dismissed', 'bowling_points']].head(10)

### Fielding Points Calculation

Fielding points reward the same moment through three lenses. `calculate_fielding_points(row)` reads the `dismissal_kind` column — the reason the ball ended — and awards **+8** for a catch, **+12** for a stumping, and **+12** for a run-out. Crucially, dismissals that earn *bowling* points (bowled, lbw, caught-and-bowled) are left alone here: the run-out and stumping reside in the fielding budget precisely because the bowler did not earn them.

The function uses an `if` / `elif` chain keyed on `dismissal_kind`, which only works because Step 2 filled blank dismissals with `'None'` — so `elif` never mis-fires on a missing marker. That dependency is the quiet architecture lesson: **every feature-engineering step relies on the cleaning step before it**, and the notebook announces that reliance by filling before scoring.

Once applied, the notebook has per-ball batting, bowling, and fielding points ready to be summed into each ball's total. The same function is then reused across every ball through `apply`, so one tested rule drives the whole column instead of a hand-typed score per row.


In [ ]:
# Calculate fielding pointsdef calculate_fielding_points(row):    points = 0    # Catch    if row['dismissal_kind'] == 'caught':        points += 8    # Stumping    elif row['dismissal_kind'] == 'stumped':        points += 12    # Run out    elif row['dismissal_kind'] == 'run out':        points += 12    return points# Apply functiondeliveries_clean['fielding_points'] = deliveries_clean.apply(calculate_fielding_points, axis=1)print("Fielding points calculated!")

### Total Dream11 Points

The three per-ball point streams converge in one clean addition: `deliveries_clean['total_points'] = batting_points + bowling_points + fielding_points`. The complexity is deliberately *not* here — engineering was written in small, audited functions, so the total is a plain column-wise sum, not another pile of rules.

Why it matters: `total_points` is the unit-level currency of the entire downstream analysis. From here, everything the pipeline builds — player-match summaries, overall career stats, the loaded tables — is an aggregation over this one honest column. If any earlier rule had been wrong, it would surface here and poison every aggregate; because each component was checked row-by-row at creation, the sum inherits that trust.

This is the ideal shape for a computed column: **derive, visualize, then aggregate**. The notebook prints ten rows of `batting_points`, `bowling_points`, `fielding_points`, `total_points` side by side — a spot-check that turns "the sum exists" into "the sum is right".


In [ ]:
# Calculate total points per balldeliveries_clean['total_points'] = (    deliveries_clean['batting_points'] +    deliveries_clean['bowling_points'] +    deliveries_clean['fielding_points'])print("Total points calculated!")deliveries_clean[['batsman', 'bowler', 'batting_points', 'bowling_points', 'fielding_points', 'total_points']].head(10)

## Step 4 — Player Match Summary

Per-ball points become per-player-per-match rows with one aggregate: `deliveries_clean.groupby(['match_id', 'batsman']).agg(...)`. The `agg` call names the columns explicitly — `runs` summed, `balls_faced` counted, `fours` and `sixes` counted with lambdas, `batting_pts` summed — so the resulting table is readable as a sentence, not a mystery.

Three enrichments follow. **Strike rate**: `runs / balls_faced * 100`, rounded. **Milestone bonuses**: +4 / +8 / +16 when `runs` crosses 30 / 50 / 100. **Duck penalty**: rows where a batsman scored 0 and was dismissed are counted per match and charged −2 each. All three are stacked into `final_batting_points`.

The lesson is aggregation algebra: `groupby(...).agg(...)` collapses grain by defining *what each new row means*. Every column in the summary is either something summed, something counted, or something derived from the sum — and the shape is printed so the collapse is verified, not assumed. When you later join batting and bowling summaries, the shared key (`match_id`, `player`) picks the same grain across tables, so no half-aggregated data leaks through the join.


In [ ]:
# Create player match summaryplayer_match = deliveries_clean.groupby(['match_id', 'batsman']).agg(    runs=('batsman_runs', 'sum'),    balls_faced=('ball', 'count'),    fours=('batsman_runs', lambda x: (x == 4).sum()),    sixes=('batsman_runs', lambda x: (x == 6).sum()),    batting_pts=('batting_points', 'sum')).reset_index()# Calculate strike rateplayer_match['strike_rate'] = (player_match['runs'] / player_match['balls_faced'] * 100).round(2)# Add milestone bonusesplayer_match['milestone_bonus'] = 0player_match.loc[player_match['runs'] >= 30, 'milestone_bonus'] += 4player_match.loc[player_match['runs'] >= 50, 'milestone_bonus'] += 8player_match.loc[player_match['runs'] >= 100, 'milestone_bonus'] += 16# Duck penalty (0 runs and out)ducks = deliveries_clean[    (deliveries_clean['batsman_runs'] == 0) &    (deliveries_clean['player_dismissed'] != 'None')].groupby(['match_id', 'batsman']).size().reset_index(name='duck_count')player_match = player_match.merge(ducks, on=['match_id', 'batsman'], how='left')player_match['duck_count'] = player_match['duck_count'].fillna(0)player_match['duck_penalty'] = player_match['duck_count'] * -2# Final batting pointsplayer_match['final_batting_points'] = (    player_match['batting_pts'] +    player_match['milestone_bonus'] +    player_match['duck_penalty'])print(f"Player match summary shape: {player_match.shape}")player_match.head()

## Step 5 — Bowling Summary

Bowling gets its own summary table via the mirror-image aggregate: `deliveries_clean.groupby(['match_id', 'bowler']).agg(...)` counting balls bowled, summing runs conceded, counting wickets (`player_dismissed != 'None'`), and totalling wides and no-balls.

Then the derived stats arrive. **Overs**: balls divided by six. **Economy**: runs conceded per over, rounded — the number that will gate the bonus. The notebook then layers Dream11's bowling rewards: an **economy bonus** (under 5 = +6, 5–6 = +4, 6–7 = +2) and a **wicket-haul bonus** (+4 / +8 / +16 at 3 / 4 / 5+ wickets), folded into `final_bowling_points`.

Note the discipline in the `loc` assignments — `bowling_match.loc[bowling_match['economy'] < 5, 'economy_bonus'] = 6` — masked updates that touch only qualifying rows. That is how feature engineering stays surgical: define the condition, update the slice, leave the rest untouched. Two details shield the arithmetic: balls are divided by six in one expression so overs stay float-precise, and every bonus column is seeded to zero before the masked updates so unpicked rows contribute nothing by default.


In [ ]:
# Create bowling summarybowling_match = deliveries_clean.groupby(['match_id', 'bowler']).agg(    balls_bowled=('ball', 'count'),    runs_conceded=('total_runs', 'sum'),    wickets=('player_dismissed', lambda x: (x != 'None').sum()),    wides=('wide_runs', 'sum'),    noballs=('noball_runs', 'sum'),    bowling_pts=('bowling_points', 'sum')).reset_index()# Calculate oversbowling_match['overs'] = (bowling_match['balls_bowled'] / 6).round(2)# Calculate economy ratebowling_match['economy'] = (bowling_match['runs_conceded'] / bowling_match['overs']).round(2)# Economy bonusbowling_match['economy_bonus'] = 0bowling_match.loc[bowling_match['economy'] < 5, 'economy_bonus'] = 6bowling_match.loc[(bowling_match['economy'] >= 5) & (bowling_match['economy'] < 6), 'economy_bonus'] = 4bowling_match.loc[(bowling_match['economy'] >= 6) & (bowling_match['economy'] < 7), 'economy_bonus'] = 2# Wicket haul bonusesbowling_match['haul_bonus'] = 0bowling_match.loc[bowling_match['wickets'] >= 3, 'haul_bonus'] += 4bowling_match.loc[bowling_match['wickets'] >= 4, 'haul_bonus'] += 8bowling_match.loc[bowling_match['wickets'] >= 5, 'haul_bonus'] += 16# Final bowling pointsbowling_match['final_bowling_points'] = (    bowling_match['bowling_pts'] +    bowling_match['economy_bonus'] +    bowling_match['haul_bonus'])print(f"Bowling match summary shape: {bowling_match.shape}")bowling_match.head()

## Step 6 — Merge with Match Data

Separate summaries are useful; enriched summaries are *valuable*. This cell joins the player-match table with match metadata: `player_summary = player_match.merge(matches_clean[['id', 'season', 'city', 'date', 'team1', 'team2', 'winner']], left_on='match_id', right_on='id', how='left')`.

Read the join carefully — it is the classic pandas relationship: `left_on='match_id'` says the left table's match key, `right_on='id'` says the right table's key, and the column lists bound the right side to exactly the match facts that matter, so the result does not balloon with redundant columns. `how='left'` keeps every player-match row even if a match reference is missing.

Two final touches complete the engineer phase: a flat **playing XI bonus** of +4 per row and the headline `dream11_points = final_batting_points + playing_xi_bonus`. The merge is the moment the pipeline stops being two parallel analyses and becomes one joinable dataset — which is, after all, the promise of Extract followed by Transform.


In [ ]:
# Add match details to player summaryplayer_summary = player_match.merge(    matches_clean[['id', 'season', 'city', 'date', 'team1', 'team2', 'winner']],    left_on='match_id',    right_on='id',    how='left')# Add playing XI bonus (4 points for each player in the match)player_summary['playing_xi_bonus'] = 4# Final Dream11 pointsplayer_summary['dream11_points'] = (    player_summary['final_batting_points'] +    player_summary['playing_xi_bonus'])print(f"Final player summary shape: {player_summary.shape}")player_summary.head()

## Step 7 — Overall Player Stats

The last transform collapses the whole project into a career table: `player_summary.groupby('batsman').agg(...)` totals each player's matches (via `nunique` on `match_id`), runs, balls faced, fours, sixes, and Dream11 points, with an average Dream11 points per match. A final derived column, `overall_strike_rate = total_runs / total_balls * 100`, rounds out the picture, and the sort on `total_runs, ascending=False` crowns the leaderboard.

This is where the dream11 scoring proves its worth: totals and career averages are aggregates over the honest `total_points` column reconstructed upstream, and `avg_dream11_points` is a genuinely useful metric — points a player adds per match he plays.

The shape of the whole transform is now visible: clean (Steps 1–2) → score (Step 3) → per-match summary (Steps 4–5) → join context (Step 6) → compress to career (Step 7). Every stage prints its shape, and every stage feeds the next. The data is now ready to leave the notebook for a database.


In [ ]:
# Calculate overall player statisticsoverall_stats = player_summary.groupby('batsman').agg(    total_matches=('match_id', 'nunique'),    total_runs=('runs', 'sum'),    total_balls=('balls_faced', 'sum'),    total_fours=('fours', 'sum'),    total_sixes=('sixes', 'sum'),    total_dream11_points=('dream11_points', 'sum'),    avg_dream11_points=('dream11_points', 'mean')).reset_index()# Calculate overall strike rateoverall_stats['overall_strike_rate'] = (    overall_stats['total_runs'] / overall_stats['total_balls'] * 100).round(2)# Sort by total runsoverall_stats = overall_stats.sort_values('total_runs', ascending=False)print(f"Overall stats shape: {overall_stats.shape}")overall_stats.head(10)

## Phase 3 — Load

Load is the payoff phase: writing the transformed DataFrames into the **AWS RDS** MySQL destination so the data outlives the notebook. The destination tables mirror the transform's own hierarchy — `matches`, `deliveries`, `dream11_points` (the per-match scoring), and `player_stats` (the career table).

Two practical realities make this phase different from the others. **Credentials**: loading into a real cloud database requires the RDS endpoint, user, password, database name and port — which the setup step captured and which the code will read into a config. **Grain**: the `deliveries` table is by far the largest and is streamed *in chunks* with `to_sql(..., if_exists='append'/'replace')`, so a big load neither hammers the connection nor dies on one giant transaction.

The philosophical point is worth stating: after Load, the analysis is no longer tied to Jupyter. Any client with the connection string — a dashboard, a model training script, a colleague's SQL client — can query the same tables. That independence is the entire reason ETL exists.


## Database Connection Setup — Option 1 (mysql-connector)

The first connection path uses the official driver directly. `mysql.connector.connect(**DB_CONFIG)` takes a config dictionary whose keys mirror the RDS setup: `host` (the RDS endpoint), `user`, `password`, `database`, and `port` — with placeholders waiting for the values the setup step captured.

The code wraps the call in `try` / `except` and prints `Connection successful!` or `Connection failed:` with the exception — a deliberate first smoke test. Notice it does *not* try to run a query; it simply proves the driver, credentials, network, and database all agree. The connection is opened purely to be verified and closed.

Why bother with this step at all when SQLAlchemy is coming next? Because failing fast at the *wire* level isolates failures: if raw `mysql-connector` cannot connect, the problem is credentials or networking, not ORM plumbing. A clean driver-level test makes every later `to_sql` failure attributable to the pandas layer, not the database.


In [ ]:
# Database connection details# Replace with your actual RDS credentialsDB_CONFIG = {    'host': 'your-rds-endpoint.rds.amazonaws.com',  # Your RDS endpoint    'user': 'admin',                                  # Master username    'password': 'your_password',                      # Master password    'database': 'ipl_db',                             # Database name    'port': 3306                                      # Default MySQL port}# Test connectiontry:    conn = mysql.connector.connect(**DB_CONFIG)    print("Connection successful!")    conn.close()except Exception as e:    print(f"Connection failed: {e}")

### Database Connection Setup — Option 2 (SQLAlchemy)

The recommended path uses **SQLAlchemy** — the engine layer that `to_sql` and `read_sql` natively understand. `create_engine(f'mysql+pymysql://{user}:{password}@{host}:{port}/{db}')` composes one URL from the RDS pieces: the scheme `mysql+pymysql` says *use the MySQL dialect driven by pymysql*; the host is the RDS endpoint; and the rest is user, password, port, and database name.

The engine string is the single most reused object in the Load and Query phases — every `to_sql` writes through it and every `pd.read_sql(query, con=engine)` reads through it, so composing it once (in a cell clearly marked with placeholder values) keeps credentials in one editable place.

Preferring SQLAlchemy over raw connectors is the pragmatic trade: pandas talks to engines, not drivers, so you can swap MySQL for PostgreSQL or Aurora later without rewriting a single `to_sql`. abstracting the database behind one string is why this option is labelled *recommended*. The entry barrier stays low too: the engine is built and tested once, and any connection error surfaces right where the credential placeholders live.


In [ ]:
# SQLAlchemy connection (recommended for pandas)# Replace with your actual RDS credentialsRDS_ENDPOINT = 'your-rds-endpoint.rds.amazonaws.com'RDS_USER = 'admin'RDS_PASSWORD = 'your_password'RDS_DB = 'ipl_db'RDS_PORT = 3306# Create engineengine = create_engine(    f'mysql+pymysql://{RDS_USER}:{RDS_PASSWORD}@{RDS_ENDPOINT}:{RDS_PORT}/{RDS_DB}')print("SQLAlchemy engine created!")

## Create Tables in RDS

Before any `to_sql`, the destination schema must exist — and this cell writes it with pure SQL: four `CREATE TABLE IF NOT EXISTS` statements executed through the engine. `matches` keys on `id INT PRIMARY KEY` with date, teams, toss, result and venue columns; `deliveries` carries the per-ball grain with `match_id`, batsman, bowler, and the run/dismissal fields; `dream11_points` stores the per-match scoring; `player_stats` keys on `batsman` with the career totals.

Notice the SQL is *declared*, not run blind — it lives in a string explicitly framed as "create tables", so the whole schema is reviewable before execution. `IF NOT EXISTS` keeps it idempotent: re-running the notebook does not error when the tables already exist.

This is the professional rhythm of Load: **define the schema before you stream the data**. The columns here were chosen to match DataFrame dtypes (INT, VARCHAR, DATE, FLOAT), so `to_sql` can map cleanly, and any analyst with a SQL client gains an immediately queryable catalogue of the pipeline's output.


In [ ]:
# Create tables using SQLcreate_tables_sql = (    "-- Matches table\n"    "CREATE TABLE IF NOT EXISTS matches (\n"    "    id INT PRIMARY KEY,\n"    "    season INT,\n"    "    city VARCHAR(100),\n"    "    date DATE,\n"    "    team1 VARCHAR(100),\n"    "    team2 VARCHAR(100),\n"    "    toss_winner VARCHAR(100),\n"    "    toss_decision VARCHAR(20),\n"    "    result VARCHAR(20),\n"    "    dl_applied INT,\n"    "    winner VARCHAR(100),\n"    "    win_by_runs INT,\n"    "    win_by_wickets INT,\n"    "    player_of_match VARCHAR(100),\n"    "    venue VARCHAR(200),\n"    "    umpire1 VARCHAR(100),\n"    "    umpire2 VARCHAR(100)\n"    ");\n\n"    "-- Deliveries table\n"    "CREATE TABLE IF NOT EXISTS deliveries (\n"    "    match_id INT,\n"    "    inning INT,\n"    "    batting_team VARCHAR(100),\n"    "    bowling_team VARCHAR(100),\n"    "    `over` INT,\n"    "    ball INT,\n"    "    batsman VARCHAR(100),\n"    "    non_striker VARCHAR(100),\n"    "    bowler VARCHAR(100),\n"    "    is_super_over INT,\n"    "    wide_runs INT,\n"    "    bye_runs INT,\n"    "    legbye_runs INT,\n"    "    noball_runs INT,\n"    "    penalty_runs INT,\n"    "    batsman_runs INT,\n"    "    extra_runs INT,\n"    "    total_runs INT,\n"    "    player_dismissed VARCHAR(100),\n"    "    dismissal_kind VARCHAR(50),\n"    "    fielder VARCHAR(100)\n"    ");\n\n"    "-- Dream11 points table\n"    "CREATE TABLE IF NOT EXISTS dream11_points (\n"    "    match_id INT,\n"    "    batsman VARCHAR(100),\n"    "    runs INT,\n"    "    balls_faced INT,\n"    "    fours INT,\n"    "    sixes INT,\n"    "    strike_rate FLOAT,\n"    "    dream11_points INT,\n"    "    season INT,\n"    "    team VARCHAR(100)\n"    ");\n\n"    "-- Player overall stats table\n"    "CREATE TABLE IF NOT EXISTS player_stats (\n"    "    batsman VARCHAR(100) PRIMARY KEY,\n"    "    total_matches INT,\n"    "    total_runs INT,\n"    "    total_balls INT,\n"    "    total_fours INT,\n"    "    total_sixes INT,\n"    "    overall_strike_rate FLOAT,\n"    "    total_dream11_points INT,\n"    "    avg_dream11_points FLOAT\n"    ");")print("SQL tables defined!")

## Load Data to RDS

Now the four tables stream from pandas to the cloud. `matches_clean.to_sql('matches', con=engine, if_exists='replace', index=False)` is the model call: give the table name, hand over the engine, replace-if-it-exists for idempotent re-runs, and drop the pandas index so the table's own primary key stays authoritative.

`deliveries` is the one large table, and its load shows the chunked pattern: `for i in range(0, len(deliveries_clean), chunk_size)` slices the frame in blocks of 10,000, and each slice writes with `if_exists='replace'` on the first chunk and `'append'` thereafter — one big table loaded in bite-size transactions, each announced by a printed chunk counter.

The `dream11_points` and `player_stats` tables follow the same recipe, and every load prints its row count so success is visible. The lesson of Load is verification at the seam: each `to_sql` states *what* it loaded and *how many rows* — because a load you cannot count is a load you cannot trust.


In [ ]:
# Load matches dataprint("Loading matches data...")matches_clean.to_sql(    'matches',    con=engine,    if_exists='replace',    index=False)print(f"Loaded {len(matches_clean)} matches")

In [ ]:
# Load deliveries data (in chunks for large datasets)print("Loading deliveries data...")chunk_size = 10000for i in range(0, len(deliveries_clean), chunk_size):    chunk = deliveries_clean.iloc[i:i+chunk_size]    chunk.to_sql(        'deliveries',        con=engine,        if_exists='append' if i > 0 else 'replace',        index=False    )    print(f"  Loaded chunk {i//chunk_size + 1}")print(f"Loaded {len(deliveries_clean)} deliveries")

In [ ]:
# Load dream11 pointsprint("Loading Dream11 points...")player_summary[['match_id', 'batsman', 'runs', 'balls_faced', 'fours', 'sixes',                'strike_rate', 'dream11_points', 'season', 'team1']].to_sql(    'dream11_points',    con=engine,    if_exists='replace',    index=False)print(f"Loaded {len(player_summary)} Dream11 records")

In [ ]:
# Load overall player statsprint("Loading player stats...")overall_stats.to_sql(    'player_stats',    con=engine,    if_exists='replace',    index=False)print(f"Loaded {len(overall_stats)} player stats")

## Querying Data from AWS RDS

The pipeline completes a full circle here: instead of reading DataFrames from CSVs, this section **reads them back from the cloud database** and proves the load worked. `pd.read_sql(query, con=engine)` turns SQL directly into a DataFrame, and the queries put the transform's product to work.

The four analytical queries are the pipeline's audition. **Top batsmen** — `ORDER BY total_runs DESC LIMIT 10` on `player_stats`. **Top Dream11 scorers** — career totals plus `avg_points_per_match` via `ROUND(..., 2)`. **Best economy bowlers** — the hard one: a nested subquery computes overs and economy per match (requiring `COUNT(*) >= 12`, about two overs), then the outer query demands at least 50 career overs and ranks by economy ascending. **Matches per season** — a simple `GROUP BY season` with distinct winners.

Read these as the return on the entire notebook: Extract + Transform produced tables, and Load made them *queryable* — raw SQL now reaches the same insights the transform computed, because the database took custody of the analysis.


In [ ]:
# Read data back from RDSquery = "SELECT * FROM player_stats ORDER BY total_runs DESC LIMIT 10"top_batsmen = pd.read_sql(query, con=engine)top_batsmen

In [ ]:
# Top Dream11 point scorersquery = (    "SELECT batsman, total_matches, total_runs, total_dream11_points, "    "ROUND(avg_dream11_points, 2) as avg_points_per_match "    "FROM player_stats "    "ORDER BY total_dream11_points DESC "    "LIMIT 10")top_dream11 = pd.read_sql(query, con=engine)top_dream11

In [ ]:
# Best economy bowlers (minimum 50 overs)query = (    "SELECT bowler, COUNT(DISTINCT match_id) as matches, "    "SUM(wickets) as total_wickets, "    "ROUND(AVG(economy), 2) as avg_economy "    "FROM ("    "SELECT match_id, bowler, "    "SUM(total_runs) as runs_conceded, "    "COUNT(*)/6 as overs, "    "SUM(total_runs)/(COUNT(*)/6) as economy, "    "SUM(CASE WHEN player_dismissed != 'None' THEN 1 ELSE 0 END) as wickets "    "FROM deliveries "    "GROUP BY match_id, bowler "    "HAVING COUNT(*) >= 12"    ") subq "    "GROUP BY bowler "    "HAVING SUM(overs) >= 50 "    "ORDER BY avg_economy ASC "    "LIMIT 10")best_economy = pd.read_sql(query, con=engine)best_economy

In [ ]:
# Matches per seasonquery = (    "SELECT season, COUNT(*) as total_matches, "    "COUNT(DISTINCT winner) as unique_winners "    "FROM matches "    "GROUP BY season "    "ORDER BY season")season_stats = pd.read_sql(query, con=engine)season_stats

## ETL Pipeline Summary

The pipeline closed its loop: **Extract** loaded `matches.csv` and `deliveries.csv` with `read_csv`, **Transform** cleaned missingness, scored every ball with the Dream11 system (batting, bowling, fielding, and total points), aggregated per-player-per-match and career summaries with match context and bonuses, and **Load** created four RDS tables — `matches`, `deliveries`, `dream11_points`, `player_stats` — and streamed them to AWS via SQLAlchemy. The finale then queried the database back: top batsmen, top Dream11 scorers, best economy bowlers, and per-season counts.

The transferable lessons are the architecture. ETL is a *phase-disciplined* pipeline: clean before you score, score before you aggregate, aggregate before you load, and verify every stage with a printed shape. `to_sql` / `read_sql` through one SQLAlchemy engine let pandas and MySQL meet cleanly, and `if_exists` discipline makes the whole build re-runnable.

What you leave with is more than working code — it is the habit of treating *the database* as the real destination of any analysis, so the work survives notebooks, servers, and colleagues.
